# 第 28 章 · 训练里用 compile 与 CUDA Graphs：消灭碎活和启动开销

本章回答：训练脚本里 `torch.compile` 套在哪一层、怎么避免重编译？什么叫 kernel 启动开销，CUDA Graph 怎么把它消掉？用 CUDA Graph 有哪些限制？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/28-compile-graphs.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch13「区域编译」（`ch13.md` 第 903–920 行）、「编译模式及其权衡」（第 826–902 行）、「使用 CUDA 图减少内核启动开销」（第 1260–1376 行：捕获第 1266 行、重放第 1314 行、最佳实践第 1331 行）；AISPE ch14 §14.1.5 动态形状（`ch14.md` 第 442–537 行）、§14.2.2 最小化重编译（第 885 行）；本讲义 [第 6 章](06-compile-stack.md)；本章 CPU 计时是本机 torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import re
import time

from torch._inductor.utils import run_and_get_code
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 40 个小算子 | eager vs compile | `per_call_us(...)` | 约 29–31 µs vs 约 4 µs |
| 第一步 融合 | — | `fused_names(many_small_ops, x)` | `cpp_fused_add_mul_0` |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `fused_names`**（来自 `ch28_compile_graphs.py`）：下面要用到的小工具。

In [ ]:
def fused_names(fn, x) -> set:
    torch._dynamo.reset()
    _, code = run_and_get_code(torch.compile(fn), x)
    return set(re.findall(r"cpp_fused\w*", code[0]))

**函数 `many_small_ops`**（来自 `ch28_compile_graphs.py`）：下面要用到的小工具。

In [ ]:
def many_small_ops(x):
    for _ in range(20):
        x = x * 1.01 + 0.01
    return x

**函数 `per_call_us`**（来自 `ch28_compile_graphs.py`）：下面要用到的小工具。

In [ ]:
def per_call_us(fn, x, n: int = 2000) -> float:
    for _ in range(50):
        fn(x)
    t = time.perf_counter()
    for _ in range(n):
        fn(x)
    return (time.perf_counter() - t) / n * 1e6

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
x = torch.randn(64)
cf = torch.compile(many_small_ops)

In [ ]:
print("\n第一步 40 个小算子：eager", round(per_call_us(many_small_ops, x), 1), "µs，compile",
      round(per_call_us(cf, x), 1), "µs")
print("融合后的 kernel：", fused_names(many_small_ops, x))

## 3. 代码里才看得到的

1. **计时数字每次略有不同**：测试只核对 compile 至少快一倍。

## 4. 自己改一改

- 把循环次数改成 200，看 eager 和 compile 各怎么涨。
- 把 x 改成 100 万个数，看差距是不是变小（计算占比变大）。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 小算子多时下单比做菜贵，compile 融合减单数；
> 大模型按块区域编，同构只编一次，变长输入补齐分桶；
> CUDA Graph 录一次套餐号，形状地址要固定、不能同步和新分配；reduce-overhead 自动录；细条加空隙就该上。